In [6]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "nibabel", "scipy"])

0

In [8]:
import nibabel as nib
import numpy as np
import scipy.ndimage as ndi
import os

in_dir = r"D:\ADNI\masks_hippo"
clean_dir = r"D:\ADNI\clean_masks_hippo"
os.makedirs(clean_dir, exist_ok=True)

labels = [1, 2]  # left / right

for fname in os.listdir(in_dir):
    if not fname.endswith("_hipp.nii.gz"):
        continue

    in_path = os.path.join(in_dir, fname)
    print(f"\n[ Processing: {in_path} ]")

    img = nib.load(in_path)
    data = img.get_fdata().astype(np.int16)
    affine = img.affine
    header = img.header

    clean = np.zeros_like(data, dtype=np.int16)

    for lab in labels:
        mask = (data == lab)
        if mask.sum() == 0:
            print(f"[경고] label {lab} voxel 없음, 건너뜀")
            continue

        labeled, n_comp = ndi.label(mask)
        print(f"label {lab}: {n_comp} connected components")

        counts = np.bincount(labeled.ravel())
        counts[0] = 0
        keep_id = counts.argmax()
        print(f"label {lab}: keep component {keep_id} (voxels = {counts[keep_id]})")

        clean[labeled == keep_id] = lab

    if fname.endswith(".nii.gz"):
        stem = fname[:-7]  # .nii.gz 제거
        out_fname = stem + "_clean.nii.gz"
    else:
        stem, ext = os.path.splitext(fname)
        out_fname = stem + "_clean" + ext

    out_path = os.path.join(clean_dir, out_fname)
    nib.save(nib.Nifti1Image(clean, affine, header), out_path)

    print("output:", out_path)
    print("unique labels (clean):", np.unique(clean))

print("\n[ ALL DONE (clean masks) ]")


=== Processing: D:\ADNI\masks_hippo\002_S_4213_20110902182731_hipp.nii.gz ===
label 1: 1 connected components
label 1: keep component 1 (voxels = 2715)
label 2: 1 connected components
label 2: keep component 1 (voxels = 2718)
output: D:\ADNI\clean_masks_hippo\002_S_4213_20110902182731_hipp_clean.nii.gz
unique labels (clean): [0 1 2]

=== Processing: D:\ADNI\masks_hippo\002_S_4225_20110921100724_hipp.nii.gz ===
label 1: 1 connected components
label 1: keep component 1 (voxels = 2963)
label 2: 1 connected components
label 2: keep component 1 (voxels = 2879)
output: D:\ADNI\clean_masks_hippo\002_S_4225_20110921100724_hipp_clean.nii.gz
unique labels (clean): [0 1 2]

=== Processing: D:\ADNI\masks_hippo\002_S_4262_20111005072430_hipp.nii.gz ===
label 1: 1 connected components
label 1: keep component 1 (voxels = 2288)
label 2: 1 connected components
label 2: keep component 1 (voxels = 2496)
output: D:\ADNI\clean_masks_hippo\002_S_4262_20111005072430_hipp_clean.nii.gz
unique labels (clean): 